# BSH Enzyme Screening — Data Preprocessing

**Objective:** Prepare BSH screening assay results (Lukowski & Dorrestein labs) for machine learning classification.

---

## Table of Contents

0. [Configuration & Imports](#0-configuration--imports)
1. [Load Reactant SMILES](#1-load-reactant-smiles)
2. [Standardize & Parse Molecules](#2-standardize--parse-molecules)
3. [Classify Bile Acids & Amines](#3-classify-bile-acids--amines)
4. [Build Feature Tables](#4-build-feature-tables)
5. [Visualize Reactants](#5-visualize-reactants)
6. [Load Heatmap Data](#6-load-heatmap-data)
7. [Combinatorial Conjugation Enumeration](#7-combinatorial-conjugation-enumeration)
8. [Product Gallery](#8-product-gallery)
9. [Product Histogram](#9-product-histogram)
10. [Heatmap Visualization](#10-heatmap-visualization)
11. [Build Labels Table](#11-build-labels-table)
12. [Load Enzyme Embeddings](#12-load-enzyme-embeddings)

## 0. Configuration & Imports

In [ ]:
# ── Configuration ────────────────────────────────────────────
from pathlib import Path

DATA_DIR   = Path("../data")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Input files
REACTANTS_XLSX     = DATA_DIR / "bsh_reactants_SMILES.xlsx"
HEATMAP_AMINES_CSV = DATA_DIR / "NEW_Stage2_BAs_amines_for_heatmap_manual.csv"
HEATMAP_SUBS_CSV   = DATA_DIR / "NEW_Stage2_BAs_subs_for_heatmap_manual.csv"
H5_PATH            = DATA_DIR / "Seqs_list_total.h5"

# Output files
FEATURES_CSV       = OUTPUT_DIR / "ml_features_conjugation_bsh_v1.csv"
FEATURES_XLSX      = OUTPUT_DIR / "ml_features_conjugation_bsh_FINAL.xlsx"
UNIQUE_PRODUCTS_XLSX = OUTPUT_DIR / "unique_conjugated_products.xlsx"
SWAP_ENUM_XLSX     = OUTPUT_DIR / "swap_enumeration_FINAL.xlsx"
SWAP_ENUM_CSV      = OUTPUT_DIR / "swap_enumeration_FINAL.csv"
HEATMAP_LONG_CSV   = OUTPUT_DIR / "ipsita_heatmap_long.csv"
EMB_NPY            = OUTPUT_DIR / "enzyme_embeddings.npy"
EMB_CSV            = OUTPUT_DIR / "enzyme_embeddings.csv"
UNSUCCESSFUL_XLSX  = OUTPUT_DIR / "unsuccessful_parents.xlsx"
UNSUCCESSFUL_CSV   = OUTPUT_DIR / "unsuccessful_parents.csv"

# Column name constants
META_COLS = {"Code", "Replicate", "filename"}
ENZYME_COL_CANDIDATES = ["Code", "Enzyme", "enzyme_id"]
REPLICATE_COL = "Replicate"
AGG_REP = "median"
DO_PER_ENZYME_SCALE = True
MIN_NONZERO_FOR_SCALE = 20
ALIGN_TO_ENUM = True

In [ ]:
# ── Imports ──────────────────────────────────────────────────
# stdlib
import os, re, math, unicodedata
from difflib import SequenceMatcher, get_close_matches

# data
import numpy as np
import pandas as pd
import h5py

# chemistry
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import (
    AllChem, Draw, Descriptors,
    rdMolDescriptors as rdmd,
    rdmolops, rdChemReactions,
)
from rdkit.Chem.AllChem import GetMorganFingerprintAsBitVect
from rdkit.Chem.rdmolops import RemoveHs, GetShortestPath
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Draw import rdMolDraw2D

# visualization
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import plotly.graph_objects as go
from IPython.display import display, Image, SVG

RDLogger.DisableLog('rdApp.warning')

## 1. Load Reactant SMILES

Loading reactant SMILES for the amine and bile acids that the Lukowski lab used for their BSH assay.

* The "Compound Type" column specifies if they are amine or bile acid.

In [ ]:
df_reactants = pd.read_excel(REACTANTS_XLSX)
df_reactants.head()

## 2. Standardize & Parse Molecules

In [ ]:
TYPE_COL = ("Compount Type" if "Compount Type" in df_reactants.columns
            else "Compound Type" if "Compound Type" in df_reactants.columns
            else None)
assert TYPE_COL is not None, "Could not find 'Compount Type' (or 'Compound Type') column."
assert "SMILES" in df_reactants.columns, "Could not find 'SMILES' column."
HAS_NAME = "Compound_Name" in df_reactants.columns

def safe_mol(smiles: str):
    try:
        return Chem.MolFromSmiles(str(smiles).strip())
    except Exception:
        return None

def standardize_mol(mol):
    """Normalize, reionize, uncharge, keep largest fragment, remove salts, sanitize, de-H."""
    if mol is None:
        return None
    try:
        rdMolStandardize.Cleanup(mol)
        mol = rdMolStandardize.Normalizer().normalize(mol)
        mol = rdMolStandardize.Reionizer().reionize(mol)
        mol = rdMolStandardize.LargestFragmentChooser().choose(mol)
        mol = rdMolStandardize.Uncharger().uncharge(mol)
        mol = SaltRemover().StripMol(mol, dontRemoveEverything=True)
        Chem.SanitizeMol(mol)
        mol = RemoveHs(mol)
        return mol
    except Exception:
        return None

def smiles_canonical(mol):
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True) if mol else None

df = df_reactants.copy()
df["Mol_raw"] = df["SMILES"].apply(safe_mol)
df["Mol"]     = df["Mol_raw"].apply(standardize_mol)
df["Standardized_SMILES"] = df["Mol"].apply(smiles_canonical)
valid_mask = df["Mol"].notnull()
print(f"Parsed {valid_mask.sum()}/{len(df)} molecules successfully.")

## 3. Classify Bile Acids & Amines

In [ ]:
# Chemistry patterns
oh_pattern   = Chem.MolFromSmarts('[CX4][OX2H]')
cooh_pattern = Chem.MolFromSmarts('C(=O)[O;H1]')

def count_core_hydroxyls(mol):
    if mol is None:
        return 0
    oh_matches   = mol.GetSubstructMatches(oh_pattern)
    cooh_matches = mol.GetSubstructMatches(cooh_pattern)
    cooh_atoms   = {idx for m in cooh_matches for idx in m}
    count = 0
    for m in oh_matches:
        oh_idx = m[1]
        if not any(len(rdmolops.GetShortestPath(mol, oh_idx, c_idx)) <= 3 for c_idx in cooh_atoms):
            count += 1
    return count

def classify_bile_acid_ignore_ketone(mol):
    if mol is None:
        return "N/A"
    core_oh = count_core_hydroxyls(mol)
    if core_oh <= 0: return "N/A"
    if core_oh == 1: return "Mono"
    if core_oh == 2: return "Di"
    return "Tri"

# Amine patterns (exclude amides)
primary_amine   = Chem.MolFromSmarts('[NX3;H2;!$(NC=O)]')
secondary_amine = Chem.MolFromSmarts('[NX3;H1;!$(NC=O)]')
tertiary_amine  = Chem.MolFromSmarts('[NX3;H0;!$(NC=O)]')

def amine_counts(mol):
    if mol is None:
        return (0, 0, 0)
    return (
        len(mol.GetSubstructMatches(primary_amine)),
        len(mol.GetSubstructMatches(secondary_amine)),
        len(mol.GetSubstructMatches(tertiary_amine)),
    )

# Molecular descriptors
_has_valence_e = hasattr(rdmd, "CalcNumValenceElectrons")
DESC_FUNCS = {
    'NumHDonors':        rdmd.CalcNumHBD,
    'NumHAcceptors':     rdmd.CalcNumHBA,
    'NumRotBonds':       rdmd.CalcNumRotatableBonds,
    'FractionCSP3':      rdmd.CalcFractionCSP3,
    'HeavyAtomCount':    rdmd.CalcNumHeavyAtoms,
    'RingCount':         rdmd.CalcNumRings,
    'AromaticRings':     rdmd.CalcNumAromaticRings,
    'AliphaticRings':    rdmd.CalcNumAliphaticRings,
}
if _has_valence_e:
    DESC_FUNCS['NumValenceElectrons'] = rdmd.CalcNumValenceElectrons

def formal_charge(mol):
    return rdmolops.GetFormalCharge(mol) if mol is not None else np.nan

def morgan_fp_bits(mol, radius=2, nBits=1024):
    if mol is None:
        return np.zeros(nBits, dtype=np.int8)
    bv = GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits)
    arr = np.zeros((nBits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(bv, arr)
    return arr

def compute_features_for_mol(mol, include_fp=True, nBits=1024):
    feats = {k: (np.nan if mol is None else DESC_FUNCS[k](mol)) for k in DESC_FUNCS}
    feats['FormalCharge']     = formal_charge(mol)
    feats['Core_OH_Count']    = (np.nan if mol is None else count_core_hydroxyls(mol))
    feats['Bile_Core_Class']  = ('N/A' if mol is None else classify_bile_acid_ignore_ketone(mol))
    p, s, t = amine_counts(mol)
    feats['PrimaryAmines']    = p
    feats['SecondaryAmines']  = s
    feats['TertiaryAmines']   = t
    if include_fp:
        fp = morgan_fp_bits(mol, radius=2, nBits=nBits)
        feats.update({f'FP_{i}': int(fp[i]) for i in range(nBits)})
    return feats

# Split by compound type
type_series = df[TYPE_COL].astype(str).str.strip()
mask_bile   = type_series.str.casefold() == "bile acid"
df_bile     = df[mask_bile].copy()
df_other    = df[~mask_bile].copy()

print(f"Total rows: {len(df)} | Bile acids: {len(df_bile)} | Other: {len(df_other)}")

## 4. Build Feature Tables

In [ ]:
def build_feature_table_from_prepared(df_subset, name_col=None, include_fp=False, nBits=1024):
    """Build feature table reusing precomputed Mol/Standardized_SMILES."""
    valid = df_subset[df_subset["Mol"].notnull()].copy()
    if "Standardized_SMILES" not in valid:
        valid["Standardized_SMILES"] = valid["Mol"].apply(smiles_canonical)
    feat_rows = [compute_features_for_mol(m, include_fp=include_fp, nBits=nBits)
                 for m in valid["Mol"]]
    features_df = pd.DataFrame(feat_rows, index=valid.index)
    keep_cols = ["Standardized_SMILES"]
    if name_col and name_col in valid.columns:
        keep_cols.append(name_col)
    meta = valid[keep_cols]
    return (pd.concat([meta, features_df], axis=1).reset_index(drop=True),
            df_subset.index[~df_subset["Mol"].notnull()].tolist())

features_other, dropped_other = build_feature_table_from_prepared(
    df_other, name_col=("Compound_Name" if HAS_NAME else None),
    include_fp=False, nBits=1024
)
features_other["Dataset"] = "other"

features_bile, dropped_bile = build_feature_table_from_prepared(
    df_bile, name_col=("Compound_Name" if HAS_NAME else None),
    include_fp=False, nBits=1024
)
features_bile["Dataset"] = "bile_acids"

print(f"Dropped rows -> Other: {len(dropped_other)} | Bile acids: {len(dropped_bile)}")

features_all = pd.concat([features_other, features_bile], ignore_index=True)
features_all.to_csv(FEATURES_CSV, index=False)
print(f"Saved: {FEATURES_CSV}")
features_all.head()

## 5. Visualize Reactants

In [ ]:
def _clip(s, n=36):
    s = str(s)
    return s if len(s) <= n else s[:n-3] + "..."

def _prep_valid(df_subset, name_col="Compound_Name"):
    valid = df_subset[df_subset["Mol"].notnull()].copy()
    if "Standardized_SMILES" not in valid:
        valid["Standardized_SMILES"] = valid["Mol"].apply(smiles_canonical)
    if name_col in valid.columns:
        legends = valid[name_col].astype(str).apply(_clip).tolist()
    else:
        legends = valid["Standardized_SMILES"].apply(_clip).tolist()
    return valid, legends

# Other substrates
valid_other, legends_other = _prep_valid(df_other, name_col="Compound_Name")
img_other = Draw.MolsToGridImage(
    valid_other["Mol"].tolist(), molsPerRow=4, subImgSize=(300, 300),
    legends=legends_other, useSVG=False
)
print("Other substrates (standardized)")
display(img_other)

# Bile acids with Mono/Di/Tri
valid_bile, _ = _prep_valid(df_bile, name_col="Compound_Name")
valid_bile["Hydroxylation_Class"] = valid_bile["Mol"].apply(classify_bile_acid_ignore_ketone)
name_or_smiles = (valid_bile["Compound_Name"].astype(str)
                  if HAS_NAME else valid_bile["Standardized_SMILES"])
valid_bile["Legend_Label"] = (valid_bile["Hydroxylation_Class"] + " - " + name_or_smiles).apply(_clip)

img_bile = Draw.MolsToGridImage(
    valid_bile["Mol"].tolist(), molsPerRow=4, subImgSize=(300, 300),
    legends=valid_bile["Legend_Label"].tolist(), useSVG=False
)
print("Bile acids (standardized, Mono/Di/Tri)")
display(img_bile)

## 6. Load Heatmap Data

* Extracting the possible product names
* There are 2 heatmaps:
  1. `NEW_Stage2_BAs_amines_for_heatmap_manual.csv` for all amine-specific products
  2. `NEW_Stage2_BAs_subs_for_heatmap_manual.csv` for other amine-containing substitutes (e.g. amino acids)

In the heatmap the **y-axis** is the protein ID and the **x-axis** is the product name (conjugated amine x bile acid).

In [ ]:
df_results_amines = pd.read_csv(HEATMAP_AMINES_CSV)
df_results_sub    = pd.read_csv(HEATMAP_SUBS_CSV)

def product_cols(df):
    """Return non-meta (product) columns for a given results table."""
    return [c for c in df.columns if c not in META_COLS]

product_cols_amines = product_cols(df_results_amines)
product_cols_sub    = product_cols(df_results_sub)
print("Amine product columns:", product_cols_amines[:10])
print("Substituent product columns:", product_cols_sub[:10])

unique_products = sorted(set(product_cols_amines) | set(product_cols_sub))
pd.DataFrame({"Unique_ProductNames": unique_products}).to_excel(UNIQUE_PRODUCTS_XLSX, index=False)
print(f"Saved {len(unique_products)} unique product names to '{UNIQUE_PRODUCTS_XLSX.name}'")

In [ ]:
unique_products

## 7. Combinatorial Conjugation Enumeration

1. Parses each product name — hydroxylation class (Mono/Di/Tri) or a position "3a7b"
2. Builds two parent pools from `features_all`: free acids (unconjugated BA set), pre-conjugates (already conjugated with taurine/glycine)
3. Hydrolysis: pre-conjugated BA -> free acid (cuts the amide)
4. Decision tree for every product type, creating a list of all possible options.

### 7a. Conjugation Detection

In [ ]:
# Bile conjugation SMARTS
amide_any      = Chem.MolFromSmarts('C(=O)N')
taurine_tail   = Chem.MolFromSmarts('NCCS(=O)(=O)[O-,$([OH])]')
glycine_tail   = Chem.MolFromSmarts('NCC(=O)[O-,$([OH])]')
cooh_carb      = Chem.MolFromSmarts('C(=O)[O-,$([OH])]')

def is_bile_conjugated(mol):
    if mol is None:
        return 0, 'NONE'
    amides = mol.GetSubstructMatches(amide_any)
    coohs  = mol.GetSubstructMatches(cooh_carb)
    if not amides or not coohs:
        return 0, 'NONE'
    cooh_c_atoms = {m[0] for m in coohs}
    candidate = []
    for a in amides:
        c_idx = a[0]
        if any(len(Chem.rdmolops.GetShortestPath(mol, c_idx, c2)) <= 3 for c2 in cooh_c_atoms):
            candidate.append(a)
    if not candidate:
        return 0, 'NONE'
    if mol.HasSubstructMatch(taurine_tail):
        return 1, 'TAU'
    if mol.HasSubstructMatch(glycine_tail):
        return 1, 'GLY'
    return 1, 'UNKNOWN'

def compute_features_with_conj(mol, include_fp=True, nBits=1024):
    base = compute_features_for_mol(mol, include_fp=include_fp, nBits=nBits)
    if "IsConjugated_SMARTS" not in base or "ConjugationType_SMARTS" not in base:
        if mol is None:
            base['IsConjugated_SMARTS'] = np.nan
            base['ConjugationType_SMARTS'] = 'N/A'
        else:
            is_c, ctype = is_bile_conjugated(mol)
            base['IsConjugated_SMARTS'] = int(is_c)
            base['ConjugationType_SMARTS'] = ctype
    return base

### 7b. Name-Based Inference

In [ ]:
# Reactants table
df_reactants = df.copy()

TYPE_COL = ("Compount Type" if "Compount Type" in df_reactants.columns
            else "Compound Type" if "Compound Type" in df_reactants.columns else None)
assert TYPE_COL is not None, "Missing 'Compount Type' (or 'Compound Type')"
HAS_NAME = "Compound_Name" in df_reactants.columns

ts = df_reactants[TYPE_COL].astype(str).str.strip().str.casefold()
mask_ba   = ts.eq("bile acid")
mask_am   = ts.str.contains("amine", na=False)

df_bile   = df_reactants[mask_ba].copy()
df_amines = df_reactants[mask_am].copy()

def _features_from_prepared(df_subset, include_fp=False, nBits=1024):
    valid = df_subset[df_subset["Mol"].notnull()].copy()
    if "Standardized_SMILES" not in valid:
        valid["Standardized_SMILES"] = valid["Mol"].apply(smiles_canonical)
    feat_rows = [compute_features_with_conj(m, include_fp=include_fp, nBits=nBits)
                 for m in valid["Mol"]]
    fx = pd.DataFrame(feat_rows, index=valid.index)
    keep_cols = ["Standardized_SMILES"]
    if HAS_NAME: keep_cols.append("Compound_Name")
    meta = valid[keep_cols]
    return pd.concat([meta, fx], axis=1).reset_index(drop=True)

features_amines = _features_from_prepared(df_amines, include_fp=False)
features_amines["Dataset"] = "amines"
features_bile   = _features_from_prepared(df_bile,   include_fp=False)
features_bile["Dataset"]   = "bile_acids"

# Name-based conjugation inference
TAURINE_NAME_ALIASES = {"TAU","TAURINE","TAURO"}
GLYCINE_NAME_ALIASES = {"GLY","GLYCINE","GLYCO"}
TAURINE_SMILES = "OS(=O)(CCN)=O"
GLYCINE_SMILES = "NCC(O)=O"
BASES = ["UDCA","CDCA","HDCA","DCA","LCA","CA"]

def _strip_accents(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")

def _clean_name(x) -> str:
    if pd.isna(x): return ""
    s = _strip_accents(str(x)).upper()
    s = re.sub(r"[\s_]+", "-", s)
    return re.sub(r"[^A-Z0-9\-]", "", s)

def _infer_conj_name(name: str):
    s = _clean_name(name)
    conj_label, conj_smiles, base_code = None, None, None
    m = re.search(r"-(GLY|TAU)$", s)
    if m:
        conj_label, conj_smiles = (("Glycine", GLYCINE_SMILES) if m.group(1)=="GLY"
                                   else ("Taurine", TAURINE_SMILES))
    elif any(a in s for a in TAURINE_NAME_ALIASES):
        conj_label, conj_smiles = "Taurine", TAURINE_SMILES
    elif any(a in s for a in GLYCINE_NAME_ALIASES):
        conj_label, conj_smiles = "Glycine", GLYCINE_SMILES
    if base_code is None:
        for b in BASES:
            if f"T{b}" in s: conj_label, conj_smiles, base_code = "Taurine", TAURINE_SMILES, b; break
            if f"G{b}" in s: conj_label, conj_smiles, base_code = "Glycine", GLYCINE_SMILES, b; break
    if base_code is None:
        for b in BASES:
            if b in s: base_code = b; break
    return conj_label, conj_smiles, base_code

if HAS_NAME and "Compound_Name" in features_bile.columns:
    conj_labels, conj_smiles, base_names = zip(*[
        _infer_conj_name(nm) for nm in features_bile["Compound_Name"].tolist()
    ])
    features_bile["Conjugated_Amine_Name"]   = list(conj_labels)
    features_bile["Conjugated_Amine_SMILES"] = list(conj_smiles)
    features_bile["Base_Acid_From_Name"]     = list(base_names)
    features_bile["Is_Conjugated_From_Name"] = features_bile["Conjugated_Amine_Name"].notna()
else:
    for c in ["Conjugated_Amine_Name","Conjugated_Amine_SMILES","Base_Acid_From_Name"]:
        features_bile[c] = None
    features_bile["Is_Conjugated_From_Name"] = False

def _final_conj_label(row):
    ctype = row.get("ConjugationType_SMARTS", "N/A")
    if ctype in ("TAU","GLY","UNKNOWN"):
        return {"TAU":"Taurine","GLY":"Glycine","UNKNOWN":"Unknown"}[ctype]
    nm = row.get("Conjugated_Amine_Name", None)
    return nm if pd.notna(nm) and nm else "None"

def _final_conj_smiles(row):
    ctype = row.get("ConjugationType_SMARTS", "N/A")
    if ctype == "TAU": return TAURINE_SMILES
    if ctype == "GLY": return GLYCINE_SMILES
    return row.get("Conjugated_Amine_SMILES", None)

features_bile["IsConjugated_Final"] = pd.to_numeric(features_bile["IsConjugated_SMARTS"], errors="coerce").fillna(0).astype(int)
features_bile.loc[
    features_bile["IsConjugated_Final"].eq(0) & features_bile["Is_Conjugated_From_Name"],
    "IsConjugated_Final"
] = 1
features_bile["ConjugationType_Final"]         = features_bile.apply(_final_conj_label,  axis=1)
features_bile["Conjugated_Amine_SMILES_Final"] = features_bile.apply(_final_conj_smiles, axis=1)

features_all = pd.concat([features_amines, features_bile], ignore_index=True)

# Save features
front_cols = [c for c in [
    "Compound_Name","Dataset","Original_SMILES","Standardized_SMILES",
    "IsConjugated_Final","ConjugationType_Final","Conjugated_Amine_SMILES_Final",
    "IsConjugated_SMARTS","ConjugationType_SMARTS",
    "Is_Conjugated_From_Name","Conjugated_Amine_Name","Conjugated_Amine_SMILES","Base_Acid_From_Name",
    "Core_OH_Count","Bile_Core_Class","PrimaryAmines","SecondaryAmines","TertiaryAmines",
] if c in features_all.columns]
desc_cols = [c for c in [
    "MolWt","LogP","TPSA","NumHDonors","NumHAcceptors","NumRotBonds",
    "FractionCSP3","HeavyAtomCount","RingCount","AromaticRings","AliphaticRings",
    "FormalCharge","NumValenceElectrons"
] if c in features_all.columns]
fp_cols = [c for c in features_all.columns if c.startswith("FP_")]
other_cols = [c for c in features_all.columns if c not in set(front_cols + desc_cols + fp_cols)]
features_all[front_cols + desc_cols + other_cols + fp_cols].to_excel(FEATURES_XLSX, index=False)
print(f"Saved features: {FEATURES_XLSX}")

### 7c. Bile Acid Pools

In [ ]:
def _greek_to_latin(s: str) -> str:
    s2 = s.replace('α','a').replace('β','b').replace('Α','a').replace('Β','b')
    s2 = re.sub(r'\balpha\b','a', s2, flags=re.I)
    s2 = re.sub(r'\bbeta\b','b', s2, flags=re.I)
    return s2

POS_RE_SIMPLE = re.compile(r'(\d+)\s*([abk])', re.I)

def extract_pos_key(s: str) -> str | None:
    if pd.isna(s) or not str(s).strip() or str(s).strip() == "nan": return None
    txt = _greek_to_latin(_strip_accents(str(s))).lower()
    txt = re.sub(r'(\d+)\s*-\s*(oxo|keto|one)\b', r'\1k', txt)
    txt = re.sub(r'(\d+)\s*(oxo|keto|one)\b',     r'\1k', txt)
    hits = POS_RE_SIMPLE.findall(txt.replace(',', ' '))
    if not hits: return None
    return ''.join(f"{n}{l.lower()}" for (n,l) in hits)

def extract_pos_tokens(s: str) -> set[str]:
    k = extract_pos_key(s)
    return set(re.findall(r'\d+[abk]', k)) if k else set()

def core_from_positional_key(pos_key: str | None) -> str | None:
    if not pos_key or (isinstance(pos_key, float) and pd.isna(pos_key)): return None
    letters = re.findall(r'\d+([abk])', pos_key.lower())
    n_oh = sum(1 for L in letters if L in ('a','b'))
    return "Tri" if n_oh>=3 else ("Di" if n_oh==2 else ("Mono" if n_oh==1 else None))

def normalize_name_salts(name):
    if pd.isna(name): return None
    s = str(name)
    s = _greek_to_latin(_strip_accents(s)).lower()
    s = re.sub(r'_m[\+\-][a-z0-9]+', '', s)
    s = re.sub(r'[_\-,]+', ' ', s)
    s = re.sub(r'\b(monohydrochloride|hydrochloride|hcl|na|k|anhydrous|monohydrate)\b', '', s)
    s = re.sub(r'^(dl|l|d)\s+', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return None if s == 'unconjugated' else s

# Amine name -> SMILES map from reactants
df_reactants2 = df_reactants.copy()
if HAS_NAME:
    df_reactants2['Norm_Name'] = df_reactants2['Compound_Name'].apply(normalize_name_salts)
else:
    df_reactants2['Norm_Name'] = None

_norm_to_smiles = (df_reactants2.dropna(subset=['Norm_Name'])
                   .groupby('Norm_Name')['SMILES']
                   .apply(lambda s: next((x for x in s if pd.notna(x) and str(x).strip()), None)).to_dict())
_norm_to_orig   = (df_reactants2.dropna(subset=['Norm_Name'])
                   .groupby('Norm_Name')['Compound_Name'].first().to_dict())
_norm_keys = list(_norm_to_orig.keys())

def best_norm_lookup(name_norm: str):
    if not name_norm: return (None, None)
    if name_norm in _norm_to_smiles:
        return _norm_to_orig[name_norm], _norm_to_smiles[name_norm]
    best, best_score = None, 0.0
    for cand in _norm_keys:
        sc = SequenceMatcher(None, name_norm, cand).ratio()
        if sc >= 0.90 and sc > best_score:
            best, best_score = cand, sc
    return (_norm_to_orig.get(best), _norm_to_smiles.get(best)) if best else (None, None)

AMINE_ALIASES = {"gaba": ["gamma-aminobutyric acid","gamma aminobutyric acid","4-aminobutyric acid","4-aminobutanoic acid","aminobutyric acid","aminobutanoic acid"]}

def best_norm_lookup_aliased(name_norm: str):
    o, s = best_norm_lookup(name_norm)
    if s: return o, s
    key = (name_norm or "").strip().lower()
    if key in AMINE_ALIASES:
        for human in AMINE_ALIASES[key]:
            alt_norm = normalize_name_salts(human)
            o2, s2 = best_norm_lookup(alt_norm)
            if s2: return o2, s2
    return (None, None)

# Build bile-acid pools
fa = features_all.copy()
for col in ["Dataset","Bile_Core_Class","Compound_Name","Standardized_SMILES",
            "IsConjugated_Final","ConjugationType_Final","Conjugated_Amine_SMILES_Final"]:
    if col not in fa.columns: fa[col] = np.nan

fa = fa[fa["Dataset"].astype(str).str.casefold().eq("bile_acids")].copy()
fa["Core_Class_norm"]   = fa["Bile_Core_Class"].astype(str).str.strip().str.lower().map({"mono":"Mono","di":"Di","tri":"Tri"})
fa["IsConj_final"]      = pd.to_numeric(fa["IsConjugated_Final"], errors="coerce").fillna(0).astype(int)
fa["ConjType_final_norm"]= fa["ConjugationType_Final"].astype(str).str.strip().str.lower().replace({"nan":"none"})
fa["ConjAmine_norm"]    = fa["ConjType_final_norm"].map({"taurine":"taurine","glycine":"glycine","unknown":"unknown","none":""}).fillna("")
fa["Positional_Key"]    = fa["Compound_Name"].apply(extract_pos_key)

ba_free = fa[fa["IsConj_final"].eq(0)].copy()
ba_pre  = fa[fa["IsConj_final"].eq(1)].copy()

ba_by_core_free = {k: d[["Compound_Name","Standardized_SMILES","Bile_Core_Class"]].reset_index(drop=True)
                   for k,d in ba_free.groupby("Core_Class_norm", dropna=False)}
ba_by_core_pre  = {k: d[["Compound_Name","Standardized_SMILES","ConjAmine_norm","Bile_Core_Class","Conjugated_Amine_SMILES_Final"]].reset_index(drop=True)
                   for k,d in ba_pre.groupby("Core_Class_norm", dropna=False)}
ba_by_pos_free  = {k: d[["Compound_Name","Standardized_SMILES","Bile_Core_Class"]].reset_index(drop=True)
                   for k,d in ba_free.dropna(subset=["Positional_Key"]).groupby("Positional_Key", dropna=False)}
ba_by_pos_pre   = {k: d[["Compound_Name","Standardized_SMILES","ConjAmine_norm","Bile_Core_Class","Conjugated_Amine_SMILES_Final"]].reset_index(drop=True)
                   for k,d in ba_pre.dropna(subset=["Positional_Key"]).groupby("Positional_Key", dropna=False)}

def _filter_pre(df_pre, am_tag):
    if df_pre.empty: return df_pre
    if am_tag in ("taurine","glycine"): return df_pre[df_pre["ConjAmine_norm"].eq(am_tag)].copy()
    if am_tag == "unknown":             return df_pre[df_pre["ConjAmine_norm"].eq("unknown")].copy()
    return df_pre.copy()

print(f"Free acids: {len(ba_free)} | Pre-conjugated: {len(ba_pre)}")
print(f"Core class pools (free): {list(ba_by_core_free.keys())}")

### 7d. Chemistry Reactions

In [ ]:
rxn_hydrolyze   = rdChemReactions.ReactionFromSmarts("[C:1](=O)[N:2]>>[C:1](=O)O")
rxn_amide_list  = [
    rdChemReactions.ReactionFromSmarts("[C:1](=O)[O;H1:2].[N;H2:3]>>[C:1](=O)[N:3]"),
    rdChemReactions.ReactionFromSmarts("[C:1](=O)[O;H1:2].[N+;H3:3]>>[C:1](=O)[N:3]"),
    rdChemReactions.ReactionFromSmarts("[C:1](=O)[O;H1:2].[N;H1:3]>>[C:1](=O)[N:3]"),
    rdChemReactions.ReactionFromSmarts("[C:1](=O)[O;H1:2].[N+;H2:3]>>[C:1](=O)[N:3]"),
]

def _mol(s):
    try:
        return Chem.MolFromSmiles(str(s)) if pd.notna(s) and str(s).strip() else None
    except Exception:
        return None

def hydrolyze_to_acid(smi):
    m = _mol(smi)
    if not m: return None
    try:
        prods = rxn_hydrolyze.RunReactants((m,))
    except Exception:
        return None
    cands = []
    for tpl in prods:
        for mol in tpl:
            try:
                Chem.SanitizeMol(mol, catchErrors=True); cands.append(mol)
            except Exception:
                pass
    if not cands: return None
    return Chem.MolToSmiles(max(cands, key=lambda x: x.GetNumAtoms()), canonical=True)

def form_amide(ba_acid_smi, amine_smi):
    m1, m2 = _mol(ba_acid_smi), _mol(amine_smi)
    if not m1 or not m2: return None
    for rxn in rxn_amide_list:
        try:
            prods = rxn.RunReactants((m1, m2))
        except Exception:
            continue
        cands = []
        for tpl in prods:
            for mol in tpl:
                try:
                    Chem.SanitizeMol(mol, catchErrors=True); cands.append(mol)
                except Exception:
                    pass
        if cands:
            try:
                return Chem.MolToSmiles(max(cands, key=lambda x: x.GetNumAtoms()), canonical=True)
            except Exception:
                continue
    return None

def _fuzzy_positional_preconj(fa_ba_pre: pd.DataFrame, am_norm: str, pos_req: str) -> pd.DataFrame:
    if not pos_req or fa_ba_pre.empty: return fa_ba_pre.iloc[0:0].copy()
    need = set(re.findall(r'\d+[abk]', pos_req.lower()))
    if not need: return fa_ba_pre.iloc[0:0].copy()
    cand = fa_ba_pre[fa_ba_pre["ConjAmine_norm"].astype(str).str.strip().str.lower() == am_norm]
    if cand.empty: return cand
    mask = cand["Compound_Name"].apply(lambda nm: need.issubset(extract_pos_tokens(nm)))
    return cand[mask].copy()

### 7e. Product Name Parsing

In [ ]:
prod_names = unique_products
prod_df = pd.DataFrame({"ProductName": prod_names})

def _parse_product_label(lbl):
    if pd.isna(lbl): return None, None, None, None
    parts  = str(lbl).strip().split('_')
    head   = parts[0] if parts else ""
    poskey = extract_pos_key(head)
    core_class = None
    if not poskey:
        t = head.strip().lower()
        if t in {"mono","di","tri"}:
            core_class = t.capitalize()
    am = "_".join(parts[1:-1]) if len(parts) > 2 else (parts[1] if len(parts)>=2 else None)
    am_norm = normalize_name_salts(am)
    rid = None
    if parts:
        m = re.search(r"\d+", parts[-1]); rid = m.group(0) if m else None
    return core_class, am_norm, rid, poskey

prod_df[["Core_Class","Amine_Text_Norm","Rhea_ID","Prod_Positional_Key"]] = (
    prod_df["ProductName"].apply(lambda x: pd.Series(_parse_product_label(x)))
)

# Amine SMILES/Type via reactants map
pat_primary   = Chem.MolFromSmarts("[NX3;H2;!$(NC=O)]")
pat_secondary = Chem.MolFromSmarts("[NX3;H1;!$(NC=O)]")
pat_tertiary  = Chem.MolFromSmarts("[NX3;H0;!$(NC=O)]")

def _guess_amine_type(smi):
    if pd.isna(smi) or not str(smi).strip(): return None
    m = Chem.MolFromSmiles(str(smi).strip())
    if not m: return None
    if m.HasSubstructMatch(pat_primary):   return "primary"
    if m.HasSubstructMatch(pat_secondary): return "secondary"
    if m.HasSubstructMatch(pat_tertiary):  return "tertiary"
    return None

def _amine_counts_from_smiles(smi):
    if pd.isna(smi) or not str(smi).strip(): return (0,0,0)
    m = Chem.MolFromSmiles(str(smi).strip())
    if not m: return (0,0,0)
    return (len(m.GetSubstructMatches(pat_primary)),
            len(m.GetSubstructMatches(pat_secondary)),
            len(m.GetSubstructMatches(pat_tertiary)))

prod_df["Amine_Match_Name"], prod_df["Amine_Match_SMILES"] = zip(
    *prod_df["Amine_Text_Norm"].apply(lambda nm: best_norm_lookup_aliased(str(nm).strip().lower() if pd.notna(nm) else ""))
)
prod_df["Amine_Type"] = prod_df["Amine_Match_SMILES"].apply(_guess_amine_type)
print(f"Parsed {len(prod_df)} products, {prod_df['Amine_Match_SMILES'].notna().sum()} with matched amine SMILES.")

### 7f. Enumeration Loop

In [ ]:
EXEMPT_AMINES = {"taurine","glycine"}
rows = []

for _, r in prod_df.iterrows():
    prod_label = r["ProductName"]
    core       = r["Core_Class"] if pd.notna(r["Core_Class"]) else None
    pos_req    = r.get("Prod_Positional_Key", None)
    if isinstance(pos_req, float) and pd.isna(pos_req): pos_req = None
    pos_core   = core_from_positional_key(pos_req)
    product_hydroxyl_class = core if core else pos_core

    am_norm = str(r["Amine_Text_Norm"]).strip().lower() if pd.notna(r["Amine_Text_Norm"]) else ""
    a_name  = r["Amine_Match_Name"]
    a_smi   = r["Amine_Match_SMILES"]
    a_type  = r["Amine_Type"]
    a_p,a_s,a_t = _amine_counts_from_smiles(a_smi)

    # choose pools by key type
    if pos_req:
        free_df = ba_by_pos_free.get(pos_req, pd.DataFrame(columns=["Compound_Name","Standardized_SMILES","Bile_Core_Class"]))
        pre_df  = ba_by_pos_pre .get(pos_req, pd.DataFrame(columns=["Compound_Name","Standardized_SMILES","ConjAmine_norm","Bile_Core_Class","Conjugated_Amine_SMILES_Final"]))
        pre_fuzzy = _fuzzy_positional_preconj(ba_pre, am_norm, pos_req) if am_norm in EXEMPT_AMINES and _filter_pre(pre_df, am_norm).empty else pd.DataFrame()
    else:
        free_df = ba_by_core_free.get(core, pd.DataFrame(columns=["Compound_Name","Standardized_SMILES","Bile_Core_Class"]))
        pre_df  = ba_by_core_pre .get(core, pd.DataFrame(columns=["Compound_Name","Standardized_SMILES","ConjAmine_norm","Bile_Core_Class","Conjugated_Amine_SMILES_Final"]))
        pre_fuzzy = pd.DataFrame()

    def _append_row(**kw):
        kw.setdefault("ProductName", prod_label)
        kw.setdefault("Core_Class", core)
        kw.setdefault("Product_Positional_Core_Class", pos_core)
        kw.setdefault("Product_Hydroxylation_Class", product_hydroxyl_class)
        kw.setdefault("Amine_Name", a_name if am_norm not in EXEMPT_AMINES else am_norm)
        kw.setdefault("Amine_SMILES", a_smi)
        kw.setdefault("Amine_Type", a_type)
        kw.setdefault("Amine_PrimaryAmines", a_p)
        kw.setdefault("Amine_SecondaryAmines", a_s)
        kw.setdefault("Amine_TertiaryAmines", a_t)
        rows.append(kw)

    # TAU/GLY fast paths
    if am_norm in EXEMPT_AMINES:
        am_smi_fixed = TAURINE_SMILES if am_norm=="taurine" else GLYCINE_SMILES

        pre_ok = _filter_pre(pre_df, am_norm)
        if pre_ok.empty and not pre_fuzzy.empty:
            pre_ok = pre_fuzzy
        for _, ba_row in pre_ok.iterrows():
            _append_row(
                Amine_SMILES=am_smi_fixed, Amine_Type=("zwitterion" if am_norm=="taurine" else "amino acid"),
                Parent_BA_Name=ba_row["Compound_Name"], Parent_BA_SMILES_Original=ba_row["Standardized_SMILES"],
                Parent_BA_Core_Class=ba_row.get("Bile_Core_Class", None),
                Parent_Preconjugated_Amine=am_norm, Acid_SMILES_Used=None,
                Swap_Source="preconjugate_as_product", Product_SMILES=ba_row["Standardized_SMILES"],
                Note="already conjugated parent; no swap",
            )

        other_tag = "glycine" if am_norm=="taurine" else "taurine"
        for _, ba_row in _filter_pre(pre_df, other_tag).iterrows():
            acid_smi = hydrolyze_to_acid(ba_row["Standardized_SMILES"])
            prod_smi = form_amide(acid_smi, am_smi_fixed) if acid_smi else None
            _append_row(
                Amine_SMILES=am_smi_fixed, Amine_Type=("zwitterion" if am_norm=="taurine" else "amino acid"),
                Parent_BA_Name=ba_row["Compound_Name"], Parent_BA_SMILES_Original=ba_row["Standardized_SMILES"],
                Parent_BA_Core_Class=ba_row.get("Bile_Core_Class", None),
                Parent_Preconjugated_Amine=other_tag, Acid_SMILES_Used=acid_smi,
                Swap_Source="hydrolyzed_from_other_conjugate", Product_SMILES=prod_smi,
                Note="swap path (other pre-conjugate -> hydrolysis -> target amine)",
            )

        for _, ba_row in free_df.iterrows():
            prod_smi = form_amide(ba_row["Standardized_SMILES"], am_smi_fixed)
            _append_row(
                Amine_SMILES=am_smi_fixed, Amine_Type=("zwitterion" if am_norm=="taurine" else "amino acid"),
                Parent_BA_Name=ba_row["Compound_Name"], Parent_BA_SMILES_Original=ba_row["Standardized_SMILES"],
                Parent_BA_Core_Class=ba_row.get("Bile_Core_Class", None),
                Parent_Preconjugated_Amine=None, Acid_SMILES_Used=ba_row["Standardized_SMILES"],
                Swap_Source="free_acid", Product_SMILES=prod_smi,
                Note=("ok" if prod_smi else "direct amidation failed"),
            )
        continue

    # general amines
    if not a_smi or (core is None and not pos_req):
        _append_row(
            Parent_BA_Name=None, Parent_BA_SMILES_Original=None, Parent_BA_Core_Class=None,
            Parent_Preconjugated_Amine=None, Acid_SMILES_Used=None,
            Swap_Source="no amine or core", Product_SMILES=None, Note="missing amine SMILES or core",
        )
        continue

    if pos_req and free_df.empty and pre_df.empty:
        _append_row(
            Parent_BA_Name=None, Parent_BA_SMILES_Original=None, Parent_BA_Core_Class=None,
            Parent_Preconjugated_Amine=None, Acid_SMILES_Used=None,
            Swap_Source="no positional parent", Product_SMILES=None,
            Note=f"positional parent required ({pos_req}) not found",
        )
        continue

    for _, ba_row in free_df.iterrows():
        prod_smi = form_amide(ba_row["Standardized_SMILES"], a_smi)
        _append_row(
            Parent_BA_Name=ba_row["Compound_Name"], Parent_BA_SMILES_Original=ba_row["Standardized_SMILES"],
            Parent_BA_Core_Class=ba_row.get("Bile_Core_Class", None),
            Parent_Preconjugated_Amine=None, Acid_SMILES_Used=ba_row["Standardized_SMILES"],
            Swap_Source="free_acid", Product_SMILES=prod_smi,
            Note=("ok" if prod_smi else "amide formation failed"),
        )

    for _, ba_row in pre_df.iterrows():
        acid_smi = hydrolyze_to_acid(ba_row["Standardized_SMILES"])
        prod_smi = form_amide(acid_smi, a_smi) if acid_smi else None
        _append_row(
            Parent_BA_Name=ba_row["Compound_Name"], Parent_BA_SMILES_Original=ba_row["Standardized_SMILES"],
            Parent_BA_Core_Class=ba_row.get("Bile_Core_Class", None),
            Parent_Preconjugated_Amine=(ba_row.get("ConjAmine_norm","") or None),
            Acid_SMILES_Used=acid_smi, Swap_Source="hydrolyzed_from_conjugate",
            Product_SMILES=prod_smi, Note=("ok" if prod_smi else "hydrolysis/amide failed"),
        )

### 7g. Save Enumeration Outputs

In [ ]:
swap_enumeration = pd.DataFrame(rows)
swap_enumeration.to_excel(SWAP_ENUM_XLSX, index=False)
swap_enumeration.to_csv(SWAP_ENUM_CSV, index=False)
print(f"Saved: {SWAP_ENUM_XLSX.name} and {SWAP_ENUM_CSV.name}")

display(features_all.head(8))
display(swap_enumeration.head(12).loc[:, [
    "ProductName","Core_Class","Product_Positional_Core_Class","Product_Hydroxylation_Class",
    "Amine_Name","Amine_Type","Amine_PrimaryAmines","Amine_SecondaryAmines","Amine_TertiaryAmines",
    "Parent_BA_Name","Parent_BA_Core_Class","Parent_Preconjugated_Amine",
    "Swap_Source","Acid_SMILES_Used","Product_SMILES","Note"
]])

## 8. Product Gallery

In [ ]:
SUCCESS_ONLY          = True
DEDUP_BY_PRODUCTNAME  = True
APPEND_CONTEXT_TO_LABEL = True
PER_PAGE              = 36
MOL_SIZE              = (250, 250)

if os.path.exists(SWAP_ENUM_XLSX):
    df_gallery = pd.read_excel(SWAP_ENUM_XLSX)
elif os.path.exists(SWAP_ENUM_CSV):
    df_gallery = pd.read_csv(SWAP_ENUM_CSV)
else:
    raise FileNotFoundError("swap_enumeration_FINAL.xlsx/csv not found.")

for c in ["ProductName", "Product_SMILES", "Amine_Name", "Parent_BA_Name", "Note"]:
    if c not in df_gallery.columns:
        df_gallery[c] = np.nan

def _not_empty(x):
    return (pd.notna(x)) and (str(x).strip() != "")

df2 = df_gallery[df_gallery["Product_SMILES"].apply(_not_empty)].copy()

if SUCCESS_ONLY:
    df2 = df2[df2["Note"].astype(str).str.lower().str.startswith("ok")]

if DEDUP_BY_PRODUCTNAME:
    df2 = df2.sort_values(["ProductName"]).drop_duplicates(subset=["ProductName"], keep="first")

df2 = df2.reset_index(drop=True)
print(f"[gallery] products to draw: {len(df2)}")

if len(df2) == 0:
    raise SystemExit("No product SMILES to draw after filtering.")

def make_mol(smiles: str):
    try:
        m = Chem.MolFromSmiles(str(smiles))
        if m is None:
            return None
        m = Chem.RemoveHs(m)
        AllChem.Compute2DCoords(m)
        return m
    except Exception:
        return None

mols = []
legends = []
bad = 0

for _, r in df2.iterrows():
    smi = str(r["Product_SMILES"])
    m = make_mol(smi)
    if m is None:
        bad += 1
        continue
    lbl = str(r["ProductName"])
    if APPEND_CONTEXT_TO_LABEL:
        a = str(r.get("Amine_Name", "") or "").strip()
        b = str(r.get("Parent_BA_Name", "") or "").strip()
        ctx = []
        if a: ctx.append(a)
        if b: ctx.append(b)
        if ctx:
            lbl = f"{lbl}\n({' + '.join(ctx)})"
    mols.append(m); legends.append(lbl)

if not mols:
    raise SystemExit("Nothing drawable (all SMILES failed to parse).")

print(f"[gallery] drawable: {len(mols)} | failed: {bad}")

n = len(mols)
n_pages = math.ceil(n / PER_PAGE)

for p in range(n_pages):
    i0 = p * PER_PAGE
    i1 = min(n, (p+1) * PER_PAGE)
    chunk_mols = mols[i0:i1]
    chunk_legs = legends[i0:i1]

    k = len(chunk_mols)
    n_cols = int(math.ceil(math.sqrt(k)))

    img = Draw.MolsToGridImage(
        chunk_mols,
        molsPerRow=n_cols,
        subImgSize=MOL_SIZE,
        legends=chunk_legs,
        useSVG=False
    )
    print(f"Page {p+1}/{n_pages}")
    display(img)

## 9. Product Histogram

Candidate parent is any bile acid attempted for a given product (free or pre-conjugated),
whereas a successful parent is one whose route yields a valid `Product_SMILES`.

In [ ]:
TOP_N  = None
HEIGHT = 600

df_hist = pd.read_excel(SWAP_ENUM_XLSX)

any_success = (
    df_hist["Product_SMILES"].notna()
    & df_hist["Product_SMILES"].astype(str).str.strip().ne("")
)
new_success = any_success & df_hist["Swap_Source"].astype(str).ne("preconjugate_as_product")

by_prod_all = df_hist.groupby("ProductName", dropna=False)["Parent_BA_Name"].nunique()
by_prod_any = df_hist[any_success].groupby("ProductName", dropna=False)["Parent_BA_Name"].nunique()
by_prod_new = df_hist[new_success].groupby("ProductName", dropna=False)["Parent_BA_Name"].nunique()

counts = (
    pd.concat([by_prod_all.rename("n_parents_all"),
               by_prod_any.rename("n_parents_success_any"),
               by_prod_new.rename("n_parents_success_new")], axis=1)
      .fillna(0).astype(int).reset_index()
)

counts = counts.sort_values("n_parents_all", ascending=False, kind="mergesort")
if TOP_N is not None:
    counts = counts.head(TOP_N)
x_order = counts["ProductName"].tolist()

fig = go.Figure()
fig.add_bar(x=counts["ProductName"], y=counts["n_parents_all"],
            name="All candidate parents", text=counts["n_parents_all"],
            textposition="outside", visible=True)
fig.add_bar(x=counts["ProductName"], y=counts["n_parents_success_any"],
            name="Successful (any product)", text=counts["n_parents_success_any"],
            textposition="outside", visible=False)
fig.add_bar(x=counts["ProductName"], y=counts["n_parents_success_new"],
            name="Successful (new chemistry)", text=counts["n_parents_success_new"],
            textposition="outside", visible=False)

fig.update_layout(
    updatemenus=[dict(
        type="buttons", direction="right",
        x=0.5, y=1.15, xanchor="center", yanchor="top",
        buttons=[
            dict(label="All candidate parents", method="update",
                 args=[{"visible":[True, False, False]},
                       {"title":"Bile-acid parents per product - All candidates"}]),
            dict(label="Successful (any product)", method="update",
                 args=[{"visible":[False, True, False]},
                       {"title":"Bile-acid parents per product - Any product"}]),
            dict(label="Successful (new chemistry)", method="update",
                 args=[{"visible":[False, False, True]},
                       {"title":"Bile-acid parents per product - New chemistry only"}]),
        ],
        showactive=True
    )],
    title="Bile-acid parents per product - All candidates",
    xaxis=dict(title="ProductName", categoryorder="array", categoryarray=x_order, tickangle=-45),
    yaxis=dict(title="Count of parent bile acids"),
    barmode="group", height=HEIGHT, margin=dict(l=60, r=30, t=90, b=160),
    dragmode="pan",
)
fig.update_traces(cliponaxis=False)
fig.show()

In [ ]:
unsuccessful = swap_enumeration.loc[swap_enumeration["Product_SMILES"].isna()].copy()

if "Note" in unsuccessful.columns:
    unsuccessful["Failure_Reason"] = unsuccessful["Note"].fillna("unknown")
else:
    unsuccessful["Failure_Reason"] = "unknown"

if "x_order" in globals():
    unsuccessful = unsuccessful[unsuccessful["ProductName"].isin(x_order)]

cols = [
    "ProductName", "Parent_BA_Name", "Parent_BA_Core_Class",
    "Parent_Preconjugated_Amine", "Swap_Source",
    "Acid_SMILES_Used", "Amine_Name", "Amine_SMILES",
    "Failure_Reason"
]
cols = [c for c in cols if c in unsuccessful.columns]

unsuccessful_view = unsuccessful.sort_values(
    by=["ProductName", "Parent_BA_Name", "Swap_Source"]
)[cols]

fail_counts = (unsuccessful.groupby("ProductName")
                           .size()
                           .reset_index(name="n_parents_unsuccessful")
                           .sort_values("n_parents_unsuccessful", ascending=False))
print(f"Total unsuccessful parent enumerations: {len(unsuccessful_view)}")
display(fail_counts.head(30).style.hide(axis="index"))
display(unsuccessful_view.head(300).style.hide(axis="index"))

unsuccessful_view.to_excel(UNSUCCESSFUL_XLSX, index=False)
unsuccessful_view.to_csv(UNSUCCESSFUL_CSV, index=False)
print(f"Saved: {UNSUCCESSFUL_XLSX.name} and {UNSUCCESSFUL_CSV.name}")

## 10. Heatmap Visualization

In [ ]:
def _row_labels(df):
    codes = df["Code"].astype(str).fillna("NA")
    if "Replicate" in df.columns:
        reps = df["Replicate"].astype(str).fillna("NA")
        return [f"{c} (R{r})" for c, r in zip(codes, reps)]
    return codes.tolist()

def _prepare_matrix(df, product_cols):
    M = df[product_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    return np.ma.masked_invalid(M)

def _plot_heatmap(df, product_cols, title, log_scale=False, vmax=None, show_max_cols=None):
    row_labels = _row_labels(df)
    data = _prepare_matrix(df, product_cols)

    cols = product_cols
    if show_max_cols is not None and len(cols) > show_max_cols:
        cols = cols[:show_max_cols]
        data = data[:, :show_max_cols]
        title = f"{title} (showing first {show_max_cols} of {len(product_cols)} columns)"

    h = min(0.35 * len(row_labels) + 2, 18)
    w = min(0.16 * len(cols) + 4, 24)
    fig, ax = plt.subplots(figsize=(w, h))

    norm = LogNorm(vmin=max(data.min() if np.isfinite(data.min()) else 1e-6, 1e-6),
                   vmax=vmax) if log_scale else None
    im = ax.imshow(data, aspect='auto', interpolation='nearest', norm=norm)

    ax.set_yticks(range(len(row_labels)))
    y_step = max(1, len(row_labels) // 40)
    ax.set_yticks(range(0, len(row_labels), y_step))
    ax.set_yticklabels([row_labels[i] for i in range(0, len(row_labels), y_step)])

    ax.set_xticks(range(len(cols)))
    x_step = max(1, len(cols) // 40)
    ax.set_xticks(range(0, len(cols), x_step))
    ax.set_xticklabels([cols[i] for i in range(0, len(cols), x_step)], rotation=90, ha='center')

    ax.set_xlabel("Product")
    ax.set_ylabel("Enzyme (Code) / Replicate")
    ax.set_title(title)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Intensity (raw)" + (" [log scale]" if log_scale else ""))

    plt.tight_layout()
    plt.show()

_plot_heatmap(df_results_amines, product_cols_amines, title="Heatmap - Amines table (raw)", log_scale=False, show_max_cols=None)
_plot_heatmap(df_results_sub,    product_cols_sub,    title="Heatmap - Substituents table (raw)", log_scale=False, show_max_cols=None)

## 11. Build Labels Table

Turning heatmap values into a table of (Enzyme, ProductName, Intensity).

In [ ]:
def _pick_col(df, cands):
    for c in cands:
        if c in df.columns: return c
    for c in df.columns:
        if any(re.sub(r'[^a-z0-9]+','', k.lower()) in re.sub(r'[^a-z0-9]+','', c.lower()) for k in cands):
            return c
    return None

def _melt(df, product_cols, enzyme_col):
    id_vars = [c for c in META_COLS if c in df.columns]
    m = df.loc[:, id_vars + product_cols].copy()
    long = m.melt(id_vars=id_vars, value_vars=product_cols,
                  var_name="ProductName", value_name="Intensity")
    if enzyme_col not in long.columns:
        raise KeyError(f"Need enzyme column '{enzyme_col}' in melted table.")
    long = long.rename(columns={enzyme_col: "Enzyme"})
    long["Enzyme"] = long["Enzyme"].astype(str)
    long["ProductName"] = long["ProductName"].astype(str)
    long["Intensity"] = pd.to_numeric(long["Intensity"], errors="coerce").fillna(0.0)
    return long[["Enzyme","ProductName","Intensity"] + ([REPLICATE_COL] if REPLICATE_COL in long.columns else [])]

def _scale_per_enzyme(df_long):
    d = df_long.copy()
    med = (d.groupby("Enzyme")["Intensity"]
             .apply(lambda s: np.median(s[s>0]) if (s>0).sum() >= MIN_NONZERO_FOR_SCALE else np.nan))
    global_med = np.nanmedian(d.loc[d["Intensity"]>0, "Intensity"]) if (d["Intensity"]>0).any() else np.nan
    scales = (global_med / med).rename("scale")
    d = d.merge(scales, left_on="Enzyme", right_index=True, how="left")
    d["Intensity"] = d["Intensity"] * d["scale"].fillna(1.0)
    return d.drop(columns=["scale"])

def build_labels_df(df_results_amines, df_results_sub, swap_enumeration=None):
    enz_A = _pick_col(df_results_amines, ENZYME_COL_CANDIDATES)
    enz_B = _pick_col(df_results_sub,    ENZYME_COL_CANDIDATES)
    if enz_A is None or enz_B is None:
        raise KeyError("Cannot find enzyme column in one of the heatmap tables.")

    prod_A = [c for c in df_results_amines.columns if c not in META_COLS]
    prod_B = [c for c in df_results_sub.columns    if c not in META_COLS]

    long_A = _melt(df_results_amines, prod_A, enzyme_col=enz_A)
    long_B = _melt(df_results_sub,    prod_B, enzyme_col=enz_B)
    labels = pd.concat([long_A, long_B], ignore_index=True)

    if DO_PER_ENZYME_SCALE:
        labels = _scale_per_enzyme(labels)

    if REPLICATE_COL and REPLICATE_COL in (set(df_results_amines.columns) | set(df_results_sub.columns)):
        if AGG_REP == "median":
            labels = (labels.groupby(["Enzyme","ProductName"], as_index=False)["Intensity"].median())
        else:
            labels = (labels.groupby(["Enzyme","ProductName"], as_index=False)["Intensity"].mean())
    else:
        labels = (labels.groupby(["Enzyme","ProductName"], as_index=False)["Intensity"].mean())

    if ALIGN_TO_ENUM and isinstance(swap_enumeration, pd.DataFrame) and "ProductName" in swap_enumeration.columns:
        valid = set(swap_enumeration["ProductName"].dropna().astype(str).unique())
        before = labels["ProductName"].nunique()
        labels = labels[labels["ProductName"].astype(str).isin(valid)].copy()
        after  = labels["ProductName"].nunique()
        if before != after:
            print(f"[align] products: {before} -> {after} (matched enumeration)")

    labels.to_csv(HEATMAP_LONG_CSV, index=False)
    print(f"[OK] Saved labels -> {HEATMAP_LONG_CSV}")
    print(f"rows={len(labels)} | enzymes={labels['Enzyme'].nunique()} | products={labels['ProductName'].nunique()}")
    return labels

labels_df = build_labels_df(df_results_amines, df_results_sub,
                            swap_enumeration=swap_enumeration if 'swap_enumeration' in globals() else None)
labels_long = labels_df.copy()
heat_long   = labels_df.copy()

> **Note:** `A0A1Y4QH48`, `Pencillin_amidase`, `nan` don't have associated embeddings.
> Need to compute them / ask where Pencillin_amidase is coming from since it has no associated ID.

In [ ]:
labels_df

## 12. Load Enzyme Embeddings

Currently working with ProtT5 embeddings.

In [ ]:
def collect_h5_datasets(h5_path):
    paths = []
    with h5py.File(h5_path, "r") as f:
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                paths.append(name)
        f.visititems(visit)
    return paths

def norm_key(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9]", "", str(s)).upper()

def last_token(code: str) -> str:
    return str(code).split("_")[-1]

all_paths = collect_h5_datasets(H5_PATH)
if not all_paths:
    raise RuntimeError(f"No datasets found in {H5_PATH}.")

bases = [p.rsplit("/", 1)[-1] for p in all_paths]
norm_bases = [norm_key(b) for b in bases]
normbase_to_path = {}
for base, normb, full in zip(bases, norm_bases, all_paths):
    normbase_to_path.setdefault(normb, full)

all_norm_bases = list(normbase_to_path.keys())

def pick_contains_match(tok_norm: str):
    hits = [b for b in all_norm_bases if tok_norm in b or b in tok_norm]
    if not hits: return None
    hits = sorted(hits, key=len)
    return normbase_to_path[hits[0]]

def pick_fuzzy_match(tok_norm: str, cutoff=0.92):
    cand = get_close_matches(tok_norm, all_norm_bases, n=1, cutoff=cutoff)
    return (normbase_to_path[cand[0]] if cand else None)

def match_dataset_for_enzyme(enzyme_id: str) -> str | None:
    full_norm = norm_key(enzyme_id)
    tok_norm  = norm_key(last_token(enzyme_id))

    if full_norm in normbase_to_path:
        return normbase_to_path[full_norm]
    if tok_norm in normbase_to_path:
        return normbase_to_path[tok_norm]
    p = pick_contains_match(tok_norm)
    if p: return p
    return pick_fuzzy_match(tok_norm, cutoff=0.92)

def fetch_and_pool(h5_path, full_path):
    with h5py.File(h5_path, "r") as f:
        arr = np.array(f[full_path])
    if arr.ndim == 2 and arr.shape[0] > 1:
        arr = arr.mean(axis=0)
    elif arr.ndim == 2 and arr.shape[0] == 1:
        arr = arr[0]
    elif arr.ndim > 2:
        arr = arr.reshape(arr.shape[-1])
    return arr.astype(np.float32)

enz_ids = sorted(heat_long["Enzyme"].astype(str).unique().tolist())
emb_rows, unmatched = [], []
for eid in enz_ids:
    path = match_dataset_for_enzyme(eid)
    if path is None:
        unmatched.append(eid); continue
    try:
        vec = fetch_and_pool(H5_PATH, path)
        emb_rows.append((eid, path, vec, int(vec.shape[-1])))
    except Exception as e:
        print(f"[Warn] failed to read {eid} at {path}: {e}")

if not emb_rows:
    raise RuntimeError("No embeddings loaded - check H5 structure and enzyme naming.")

df_enz = pd.DataFrame(emb_rows, columns=["Enzyme", "h5_path", "Embedding", "dim"])
E = np.stack(df_enz["Embedding"].values, axis=0).astype(np.float32)
d_prot = E.shape[1]
enzyme2idx = {eid: i for i, eid in enumerate(df_enz["Enzyme"].tolist())}

print(f"[OK] Embedding matrix: {E.shape} (d_prot={d_prot})")
print(f"[Match] {len(df_enz)} matched / {len(enz_ids)} enzymes in heat_long.")
if unmatched[:10]:
    print("[Unmatched] examples:", unmatched[:10])

np.save(EMB_NPY, {row.Enzyme: row.Embedding for _, row in df_enz.iterrows()})
emb_cols = [f"enz_{i}" for i in range(d_prot)]
pd.concat(
    [df_enz[["Enzyme"]].reset_index(drop=True),
     pd.DataFrame(E, columns=emb_cols)],
    axis=1
).to_csv(EMB_CSV, index=False)
print(f"Saved: {EMB_NPY.name} and {EMB_CSV.name}")

In [ ]:
df_enz.head()

The files generated with this code will now be used for the building of the model — see `BSH_conjugation_model.ipynb`.